# Task 8 — Full-Scene Traffic Sign Detection with YOLOv8

**Project:** German Traffic Sign Detection Benchmark (GTSDB)  
**Workflow:** dataset audit → annotation conversion → leakage-safe split → training → evaluation → speed benchmark → ONNX → recorded-video inference.

This is the complete working notebook. Run cells from top to bottom. Some cells deliberately stop with a clear message if the dataset mirror has a different layout or annotation format; inspect the printed audit and adjust the clearly marked configuration rather than guessing. Training metrics are generated only by running the model.

**Important:** this project performs object detection on full road-scene images (bounding boxes + class labels), not cropped-image classification. Raw data, weights and generated videos stay in Google Drive, not GitHub.

## 0. Runtime and project configuration

In Colab choose **Runtime → Change runtime type → T4 GPU** if available. CPU is supported for debugging but training will be much slower. This notebook uses a Google Drive workspace so data and checkpoints survive Colab sessions.

In [ ]:
from pathlib import Path
import os, sys, json, random, shutil, zipfile, subprocess, textwrap, time, platform
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/traffic-sign-detection")
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RUNS_DIR = PROJECT_DIR / "runs"
EXPORT_DIR = PROJECT_DIR / "exports"
VIDEO_DIR = PROJECT_DIR / "videos"
for p in [RAW_DIR, PROCESSED_DIR, RUNS_DIR, EXPORT_DIR, VIDEO_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project workspace:", PROJECT_DIR)
print("Python:", sys.version)
print("Platform:", platform.platform())

## 1. Install dependencies and check hardware

If Colab reports that a package was already imported before installation, restart the runtime once and rerun from the top.

In [ ]:
%pip -q install -U "ultralytics>=8.3,<9" opencv-python-headless pandas pyyaml pillow matplotlib seaborn scikit-learn onnx onnxruntime kaggle

import torch, cv2, ultralytics
from ultralytics import YOLO
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("OpenCV:", cv2.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU unavailable. You can continue for data preparation, but consider enabling a Colab GPU before training.")

## 2. Download the dataset

Default Kaggle dataset: `safabouguezzi/german-traffic-sign-detection-benchmark-gtsdb`.

Create a Kaggle API token in your Kaggle account (Settings → API → Create New Token), then upload the downloaded `kaggle.json` when prompted. The token is copied directly into the Colab runtime with restrictive permissions. **Never paste it into this notebook, GitHub, or this chat.**

If you already have the official GTSDB archive, set `USE_KAGGLE = False` and upload/extract it into `data/raw`.

In [ ]:
USE_KAGGLE = True
KAGGLE_DATASET = "safabouguezzi/german-traffic-sign-detection-benchmark-gtsdb"
if USE_KAGGLE:
    from google.colab import files
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    token_path = kaggle_dir / "kaggle.json"
    if not token_path.exists():
        print("Upload your Kaggle API token file (kaggle.json).")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("Expected an uploaded file named kaggle.json")
        token_path.write_bytes(uploaded["kaggle.json"])
        token_path.chmod(0o600)
    os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_dir)
    download_dir = RAW_DIR / "kaggle_download"
    download_dir.mkdir(parents=True, exist_ok=True)
    if not any(download_dir.iterdir()):
        subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(download_dir), "--unzip"], check=True)
    else:
        print("Existing Kaggle download found; skipping download.")
    token_path.unlink(missing_ok=True)
    print("Kaggle token removed from the runtime after download.")
else:
    print("Place the official extracted GTSDB files under:", RAW_DIR)

## 3. Audit the downloaded files before preprocessing

This audit is intentionally thorough. Check that the files represent **full-scene images** and include bounding-box annotations. GTSRB-style folders of individual cropped signs are not a substitute for GTSDB detection data.

In [ ]:
IMAGE_EXTS = {".ppm", ".png", ".jpg", ".jpeg", ".bmp"}
all_files = [p for p in RAW_DIR.rglob("*") if p.is_file()]
images = [p for p in all_files if p.suffix.lower() in IMAGE_EXTS]
annotation_candidates = [p for p in all_files if p.suffix.lower() in {".csv", ".txt", ".gt", ".xml"}]
readmes = [p for p in all_files if "readme" in p.name.lower() or p.name.lower().endswith("classes.txt")]

print(f"Files: {len(all_files):,} | Images: {len(images):,} | Possible annotation files: {len(annotation_candidates)}")
print("\nTop-level entries:")
for p in sorted(RAW_DIR.iterdir()):
    print(("DIR " if p.is_dir() else "FILE"), p.name)
print("\nImage extensions:", pd.Series([p.suffix.lower() for p in images]).value_counts().to_dict())
print("\nPossible annotation files:")
for p in annotation_candidates[:60]:
    print(p.relative_to(RAW_DIR), f"({p.stat().st_size:,} bytes)")
print("\nReadme/class files:")
for p in readmes[:30]:
    print(p.relative_to(RAW_DIR))
if images:
    from PIL import Image
    print("\nSample image dimensions:")
    for p in images[:10]:
        try:
            with Image.open(p) as im: print(p.relative_to(RAW_DIR), im.size, im.mode)
        except Exception as e: print("Unreadable:", p, e)

## 4. Inspect annotation samples and class documentation

Review these outputs carefully. Official GTSDB annotations are commonly semicolon-separated rows containing filename, left, top, right, bottom and class ID. Kaggle mirrors sometimes rename files or transform labels, so do not assume the format without inspecting it.

In [ ]:
for p in annotation_candidates[:20]:
    print("\n" + "="*90)
    print("FILE:", p.relative_to(RAW_DIR))
    try:
        print(p.read_text(errors="replace")[:2500])
    except Exception as e:
        print("Could not read as text:", e)
for p in readmes[:10]:
    print("\nREADME:", p.relative_to(RAW_DIR))
    print(p.read_text(errors="replace")[:7000])

## 5. Configure dataset paths and class names

After reviewing the audit, edit the values below if the mirror uses different paths. For the official GTSDB annotation format, set `ANNOTATION_FILE` to the annotation file and `IMAGE_ROOT` to the folder containing the corresponding full-scene images.

**Class names:** populate `CLASS_NAMES` from the dataset's supplied ReadMe/class mapping. Do not invent class names or assume the 43 class IDs used by GTSRB are automatically the right mapping for this benchmark. The parser checks IDs against this list.

In [ ]:
def find_first(candidates):
    return candidates[0] if candidates else None

# EDIT THESE AFTER THE AUDIT if automatic discovery selects the wrong file/folder.
ANNOTATION_FILE = find_first([p for p in annotation_candidates if p.suffix.lower() == ".csv"])
IMAGE_ROOT = RAW_DIR

# Official GTSDB has fine-grained sign IDs; use the exact supplied mapping for your archive.
# Keep this as None until you have verified the class mapping from the dataset documentation.
CLASS_NAMES = None

print("Selected annotation file:", ANNOTATION_FILE)
print("Image root:", IMAGE_ROOT)
print("Verified class names configured:", CLASS_NAMES is not None)
if ANNOTATION_FILE is None:
    print("No CSV candidate found. Inspect the audit and set ANNOTATION_FILE manually.")
if CLASS_NAMES is None:
    print("STOP before label conversion/training: set CLASS_NAMES from the supplied dataset ReadMe.")

## 6. Parse GTSDB annotations and validate bounding boxes

This parser targets the native six-column semicolon-separated GTSDB format. It supports an optional header. If the audit shows a different format, stop here and adapt the parser to the observed schema rather than silently misreading labels.

In [ ]:
def parse_gtsdb_native(annotation_path):
    rows = []
    with open(annotation_path, "r", encoding="utf-8-sig") as f:
        for line_no, raw in enumerate(f, 1):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = [x.strip() for x in line.split(";")]
            if len(parts) != 6:
                # tolerate a single header row only
                if line_no == 1 and any("filename" in x.lower() for x in parts):
                    continue
                raise ValueError(f"Line {line_no}: expected 6 semicolon-separated fields, got {len(parts)}: {line}")
            try:
                filename = parts[0]
                x1, y1, x2, y2, cls = map(int, parts[1:])
            except ValueError:
                if line_no == 1:
                    continue
                raise
            if x2 <= x1 or y2 <= y1:
                raise ValueError(f"Invalid box at line {line_no}: {parts}")
            rows.append({"filename": filename, "x1": x1, "y1": y1, "x2": x2, "y2": y2, "class_id": cls})
    return pd.DataFrame(rows)

if ANNOTATION_FILE is None:
    raise FileNotFoundError("Set ANNOTATION_FILE after inspecting the audit.")
annotations = parse_gtsdb_native(ANNOTATION_FILE)
print("Annotation rows:", len(annotations))
display(annotations.head())
print("Class IDs:", sorted(annotations.class_id.unique().tolist()))
print("Boxes per image:")
display(annotations.groupby("filename").size().describe())

## 7. Match image paths, validate IDs and inspect boxes

In [ ]:
from PIL import Image
image_lookup = {}
for p in IMAGE_ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
        image_lookup.setdefault(p.name, p)

annotations["image_path"] = annotations.filename.map(lambda n: str(image_lookup[n]) if n in image_lookup else None)
missing = annotations[annotations.image_path.isna()]
print("Rows without a matching image:", len(missing))
if len(missing):
    display(missing.head(20))
    raise FileNotFoundError("Some annotation filenames did not match images. Correct IMAGE_ROOT or filename mapping.")

if CLASS_NAMES is None:
    raise ValueError("Set CLASS_NAMES from the supplied class documentation before continuing.")
if not isinstance(CLASS_NAMES, (list, tuple)) or not CLASS_NAMES:
    raise ValueError("CLASS_NAMES must be a non-empty ordered list.")
if annotations.class_id.min() < 0 or annotations.class_id.max() >= len(CLASS_NAMES):
    raise ValueError(f"Class IDs outside configured range 0..{len(CLASS_NAMES)-1}. Recheck class mapping.")

bad = []
for row in annotations.itertuples():
    with Image.open(row.image_path) as im:
        w, h = im.size
    if row.x1 < 0 or row.y1 < 0 or row.x2 > w or row.y2 > h:
        bad.append((row.filename, (row.x1,row.y1,row.x2,row.y2), (w,h)))
print("Out-of-image boxes:", len(bad))
if bad:
    print(bad[:10])
    raise ValueError("Fix coordinate interpretation or annotation mapping before proceeding.")
print("Validated class count:", len(CLASS_NAMES))
print("Unique annotated images:", annotations.filename.nunique())

## 8. Visualize raw annotations

Confirm that boxes actually surround traffic signs. This is a critical visual quality check before training.

In [ ]:
import matplotlib.pyplot as plt
def show_annotated_samples(frame, n=6):
    names = frame.filename.drop_duplicates().sample(min(n, frame.filename.nunique()), random_state=SEED).tolist()
    fig, axes = plt.subplots(len(names), 1, figsize=(12, 5*len(names)))
    if len(names) == 1: axes = [axes]
    for ax, name in zip(axes, names):
        rows = frame[frame.filename == name]
        img = cv2.imread(rows.iloc[0].image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        for r in rows.itertuples():
            rect = plt.Rectangle((r.x1,r.y1), r.x2-r.x1, r.y2-r.y1, fill=False, linewidth=2)
            ax.add_patch(rect)
            ax.text(r.x1, max(0,r.y1-4), CLASS_NAMES[r.class_id], color="yellow", fontsize=8,
                    bbox=dict(facecolor="black", alpha=.6))
        ax.set_title(name); ax.axis("off")
    plt.tight_layout(); plt.show()

show_annotated_samples(annotations)

## 9. Create leakage-safe train/validation/test splits

If the downloaded benchmark provides an official train/test partition, preserve it and split only the training partition into train/validation. Set the two path variables below based on the audit. If the mirror contains only one annotated pool, this cell creates a deterministic image-level split and documents that it is **not** the official benchmark split.

Never split individual boxes independently: all boxes from one scene must stay together.

In [ ]:
# OPTIONAL: set these to the official training and test annotation files if supplied separately.
OFFICIAL_TRAIN_ANNOTATIONS = ANNOTATION_FILE
OFFICIAL_TEST_ANNOTATIONS = None  # set if the archive supplies a separate official test annotation file

from sklearn.model_selection import train_test_split
if OFFICIAL_TEST_ANNOTATIONS is not None:
    train_all = parse_gtsdb_native(OFFICIAL_TRAIN_ANNOTATIONS)
    test_df = parse_gtsdb_native(OFFICIAL_TEST_ANNOTATIONS)
    train_names = train_all.filename.drop_duplicates().tolist()
    tr_names, val_names = train_test_split(train_names, test_size=0.2, random_state=SEED)
    train_df = train_all[train_all.filename.isin(tr_names)].copy()
    val_df = train_all[train_all.filename.isin(val_names)].copy()
    split_note = "Official test partition preserved; validation split from official training partition."
else:
    names = annotations.filename.drop_duplicates().tolist()
    train_names, hold_names = train_test_split(names, test_size=0.30, random_state=SEED)
    val_names, test_names = train_test_split(hold_names, test_size=0.50, random_state=SEED)
    train_df = annotations[annotations.filename.isin(train_names)].copy()
    val_df = annotations[annotations.filename.isin(val_names)].copy()
    test_df = annotations[annotations.filename.isin(test_names)].copy()
    split_note = "No separate official test annotations detected/configured; deterministic 70/15/15 image-level split."
print(split_note)
print("Images:", {k: v.filename.nunique() for k,v in {"train":train_df,"val":val_df,"test":test_df}.items()})
assert set(train_df.filename).isdisjoint(val_df.filename)
assert set(train_df.filename).isdisjoint(test_df.filename)
assert set(val_df.filename).isdisjoint(test_df.filename)

## 10. Convert annotations to YOLO format and build dataset folders

YOLO detection labels use one text file per image with rows: `class_id x_center y_center width height`, normalized to 0–1. Empty-label images are retained so the detector also learns background scenes.

In [ ]:
from collections import defaultdict
import yaml

YOLO_ROOT = PROCESSED_DIR / "gtsdb"
for split in ["train","val","test"]:
    (YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

def convert_split(frame, split):
    grouped = defaultdict(list)
    path_by_name = {}
    for r in frame.itertuples():
        grouped[r.filename].append(r)
        path_by_name[r.filename] = Path(r.image_path)
    # Include every image found under IMAGE_ROOT, including negative scenes.
    selected_names = list(path_by_name)
    for name in selected_names:
        src = path_by_name[name]
        with Image.open(src) as im:
            w,h = im.size
        dst_name = f"{Path(name).stem}{src.suffix.lower()}"
        shutil.copy2(src, YOLO_ROOT/"images"/split/dst_name)
        label_path = YOLO_ROOT/"labels"/split/(Path(dst_name).stem+".txt")
        lines=[]
        for r in grouped[name]:
            xc=((r.x1+r.x2)/2)/w; yc=((r.y1+r.y2)/2)/h
            bw=(r.x2-r.x1)/w; bh=(r.y2-r.y1)/h
            lines.append(f"{r.class_id} {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}")
        label_path.write_text("\n".join(lines))
    return len(selected_names)

# Rebuild only the processed dataset, leaving raw data untouched.
if YOLO_ROOT.exists(): shutil.rmtree(YOLO_ROOT)
for split in ["train","val","test"]:
    (YOLO_ROOT/"images"/split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT/"labels"/split).mkdir(parents=True, exist_ok=True)
counts = {name:convert_split(df,name) for name,df in [("train",train_df),("val",val_df),("test",test_df)]}
print("Converted image counts:", counts)

## 11. Write YOLO dataset configuration and validate labels

In [ ]:
data_yaml = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(CLASS_NAMES),
    "names": list(CLASS_NAMES),
}
yaml_path = PROJECT_DIR/"gtsdb.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print(yaml_path.read_text())

def validate_yolo_split(split):
    image_dir=YOLO_ROOT/"images"/split
    label_dir=YOLO_ROOT/"labels"/split
    problems=[]
    for img_path in image_dir.iterdir():
        label_path=label_dir/(img_path.stem+".txt")
        if not label_path.exists():
            problems.append(f"Missing label file: {label_path}")
            continue
        for line_no,line in enumerate(label_path.read_text().splitlines(),1):
            vals=line.split()
            if len(vals)!=5:
                problems.append(f"{label_path}:{line_no}: expected 5 values")
                continue
            cls,xc,yc,bw,bh=map(float,vals)
            if int(cls)!=cls or not (0<=int(cls)<len(CLASS_NAMES)):
                problems.append(f"{label_path}:{line_no}: invalid class {cls}")
            if not all(0<=v<=1 for v in [xc,yc,bw,bh]) or bw<=0 or bh<=0:
                problems.append(f"{label_path}:{line_no}: invalid normalized box")
    return problems
for split in ["train","val","test"]:
    issues=validate_yolo_split(split)
    print(split, "issues:", len(issues))
    if issues: print(issues[:10]); raise ValueError("YOLO label validation failed.")
print("Dataset conversion and validation passed.")

## 12. Train YOLOv8

Start with YOLOv8n as a lightweight baseline. For small signs, image size 960 can help preserve detail but increases memory and inference cost. If GPU memory is insufficient, reduce `IMGSZ` to 640 or `BATCH` to 8. Checkpoints and plots are copied to Drive after training.

In [ ]:
MODEL_NAME = "yolov8n.pt"
IMGSZ = 960
EPOCHS = 100
BATCH = 8
PATIENCE = 20
DEVICE = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(yaml_path),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    patience=PATIENCE,
    device=DEVICE,
    workers=2,
    seed=SEED,
    project=str(RUNS_DIR),
    name="gtsdb_yolov8n",
    exist_ok=True,
    plots=True,
    cache=False,
    pretrained=True,
    amp=torch.cuda.is_available(),
)
BEST_WEIGHTS = RUNS_DIR/"gtsdb_yolov8n"/"weights"/"best.pt"
LAST_WEIGHTS = RUNS_DIR/"gtsdb_yolov8n"/"weights"/"last.pt"
assert BEST_WEIGHTS.exists(), f"Best checkpoint not found: {BEST_WEIGHTS}"
print("Best checkpoint:", BEST_WEIGHTS)

## 13. Evaluate on validation and held-out test data

Use validation results for model selection and confidence-threshold tuning. Run the held-out test evaluation once for final reporting. If the split above is a random split rather than the official benchmark partition, state that limitation in the final report.

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
val_metrics = best_model.val(data=str(yaml_path), split="val", imgsz=IMGSZ, device=DEVICE, plots=True)
test_metrics = best_model.val(data=str(yaml_path), split="test", imgsz=IMGSZ, device=DEVICE, plots=True)

def metric_summary(metrics):
    box=getattr(metrics,"box",None)
    if box is None: return {}
    return {
        "precision": float(box.mp),
        "recall": float(box.mr),
        "mAP50": float(box.map50),
        "mAP50-95": float(box.map),
    }
evaluation = {"validation":metric_summary(val_metrics),"test":metric_summary(test_metrics)}
print(json.dumps(evaluation,indent=2))
(PROJECT_DIR/"evaluation_metrics.json").write_text(json.dumps(evaluation,indent=2))

## 14. Review training curves and predictions

Look for overfitting, poor recall on small signs, class imbalance and localization errors. Do not rely on one aggregate metric alone.

In [ ]:
from IPython.display import display, Image as IPyImage
results_dir=RUNS_DIR/"gtsdb_yolov8n"
for filename in ["results.png","confusion_matrix.png","PR_curve.png","F1_curve.png"]:
    p=results_dir/filename
    if p.exists():
        print(filename); display(IPyImage(filename=str(p)))
preds=best_model.predict(source=str(YOLO_ROOT/"images"/"test"), imgsz=IMGSZ, conf=0.25, device=DEVICE, save=True,
                         project=str(RUNS_DIR), name="test_predictions", exist_ok=True)
print("Prediction output:", RUNS_DIR/"test_predictions")

## 15. Benchmark inference latency and FPS

This reports model inference throughput on the current Colab runtime, not end-to-end video throughput. Warm-up is excluded; timings use repeated single-image inference. Record GPU/CPU, image size, batch size and runtime because FPS is hardware-dependent.

In [ ]:
import statistics
test_images=list((YOLO_ROOT/"images"/"test").glob("*"))
if not test_images: raise RuntimeError("No test images available.")
sample_paths=test_images[:min(100,len(test_images))]
for p in sample_paths[:5]:
    best_model.predict(source=str(p), imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)
latencies=[]
for p in sample_paths:
    t0=time.perf_counter()
    best_model.predict(source=str(p), imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)
    latencies.append((time.perf_counter()-t0)*1000)
mean_ms=float(np.mean(latencies))
fps=1000/mean_ms
speed_report={
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "imgsz": IMGSZ, "batch_size":1, "images_timed":len(latencies),
    "mean_latency_ms":mean_ms, "median_latency_ms":float(np.median(latencies)),
    "p95_latency_ms":float(np.percentile(latencies,95)), "approx_fps":fps,
    "timing_scope":"single-image predict call; includes preprocessing/postprocessing overhead; excludes file decode variance and video display"
}
print(json.dumps(speed_report,indent=2))
(PROJECT_DIR/"speed_benchmark.json").write_text(json.dumps(speed_report,indent=2))

## 16. Export to ONNX and verify runtime inference

Export from the best PyTorch checkpoint. Then load the ONNX model with ONNX Runtime and compare basic output shape/execution. The exported graph's raw outputs are not directly comparable to postprocessed YOLO detections without applying the same decode/NMS pipeline, so this cell verifies successful inference rather than claiming numerical parity.

In [ ]:
onnx_path = Path(best_model.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True))
drive_onnx = EXPORT_DIR/"best.onnx"
shutil.copy2(onnx_path, drive_onnx)
import onnx
onnx_model=onnx.load(str(drive_onnx))
onnx.checker.check_model(onnx_model)
import onnxruntime as ort
providers=["CUDAExecutionProvider","CPUExecutionProvider"] if torch.cuda.is_available() else ["CPUExecutionProvider"]
session=ort.InferenceSession(str(drive_onnx),providers=providers)
input_meta=session.get_inputs()[0]
dummy=np.zeros(input_meta.shape,dtype=np.float32)
outputs=session.run(None,{input_meta.name:dummy})
print("ONNX validated:", drive_onnx)
print("Providers:",session.get_providers())
print("Input:",input_meta.name,input_meta.shape)
print("Output shapes:",[x.shape for x in outputs])

## 17. Recorded-road-video inference (webcam alternative)

You do not need a physical webcam. Upload a road-driving video that you have permission to use, or use a suitably licensed/public sample and record its source and licence. This is **prerecorded-video inference**, not live webcam testing.

The output video is saved to Drive. If the video is very large, trim it before processing.

In [ ]:
from google.colab import files
uploaded_video = files.upload()
video_names=[n for n in uploaded_video if Path(n).suffix.lower() in {".mp4",".avi",".mov",".mkv"}]
if not video_names:
    raise FileNotFoundError("Upload a supported road video (.mp4, .avi, .mov or .mkv).")
VIDEO_SOURCE=Path(video_names[0])
video_out=VIDEO_DIR/f"{VIDEO_SOURCE.stem}_detections.mp4"

CONFIDENCE_THRESHOLD=0.25
cap=cv2.VideoCapture(str(VIDEO_SOURCE))
if not cap.isOpened(): raise RuntimeError("Could not open uploaded video.")
fps_in=cap.get(cv2.CAP_PROP_FPS) or 25.0
width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
writer=cv2.VideoWriter(str(video_out),cv2.VideoWriter_fourcc(*"mp4v"),fps_in,(width,height))
frame_count=0
t_start=time.perf_counter()
while True:
    ok,frame=cap.read()
    if not ok: break
    result=best_model.predict(frame,conf=CONFIDENCE_THRESHOLD,imgsz=IMGSZ,device=DEVICE,verbose=False)[0]
    annotated=result.plot()
    writer.write(annotated)
    frame_count+=1
cap.release(); writer.release()
elapsed=time.perf_counter()-t_start
print("Saved:",video_out)
print("Frames:",frame_count,"Elapsed seconds:",round(elapsed,2))
print("End-to-end video processing FPS:",round(frame_count/elapsed,2) if elapsed else None)

## 18. Confidence-threshold review

The confidence threshold controls the precision/recall trade-off. Use validation data to compare thresholds; do not tune against the held-out test set. This quick sweep logs validation mAP/precision/recall for selected confidence settings.

In [ ]:
threshold_rows=[]
for conf in [0.10,0.15,0.20,0.25,0.30,0.40,0.50]:
    m=best_model.val(data=str(yaml_path),split="val",imgsz=IMGSZ,device=DEVICE,conf=conf,plots=False,verbose=False)
    threshold_rows.append({"confidence":conf,**metric_summary(m)})
threshold_df=pd.DataFrame(threshold_rows)
display(threshold_df)
threshold_df.to_csv(PROJECT_DIR/"confidence_threshold_validation.csv",index=False)

## 19. Save a reproducibility report

This collects the dataset split note, class mapping, package versions, training settings, evaluation metrics and measured speed. Review and supplement the dataset source/licence and any deviations before publishing results.

In [ ]:
import importlib.metadata as metadata
report={
    "project":"GTSDB full-scene traffic sign detection",
    "dataset_source":KAGGLE_DATASET if USE_KAGGLE else "Manually supplied official archive",
    "split_note":split_note,
    "seed":SEED,
    "classes":list(CLASS_NAMES),
    "images_per_split":{k:int(v.filename.nunique()) for k,v in {"train":train_df,"val":val_df,"test":test_df}.items()},
    "training":{"model":MODEL_NAME,"imgsz":IMGSZ,"epochs_requested":EPOCHS,"batch":BATCH,"device":DEVICE},
    "evaluation":evaluation,
    "speed":speed_report,
    "versions":{p:metadata.version(p) for p in ["ultralytics","torch","opencv-python-headless","onnxruntime"] if any(d.metadata["Name"].lower()==p.lower() for d in metadata.distributions())},
    "video_inference":{"input":str(VIDEO_SOURCE),"output":str(video_out),"confidence_threshold":CONFIDENCE_THRESHOLD},
    "limitations":[
        "Report whether the dataset mirror is complete and whether official benchmark splits were preserved.",
        "FPS depends on runtime hardware, image size and measurement scope.",
        "Recorded-video inference is not physical webcam validation."
    ]
}
report_path=PROJECT_DIR/"task_08_results_report.json"
report_path.write_text(json.dumps(report,indent=2))
print("Report saved:",report_path)
print(json.dumps(report,indent=2))

## 20. Final checklist and portfolio handoff

Before marking Task 8 complete:

- [ ] Confirm dataset source, completeness and licence.
- [ ] Verify the class-ID mapping against the supplied documentation.
- [ ] Inspect annotated full-scene samples and label quality.
- [ ] Confirm no image overlap between train, validation and test.
- [ ] Record the split protocol and whether the official test set was used.
- [ ] Save best weights and ONNX export in Drive or a model registry (not Git).
- [ ] Record precision, recall, mAP50, mAP50-95 and per-class results.
- [ ] Record hardware, input size, latency and FPS methodology.
- [ ] Save prediction examples and the annotated prerecorded video.
- [ ] Update the repository README with measured results only.
- [ ] Run project tests and review the GitHub PR before merging.

**Next collaboration step:** run Sections 0–4 first and share the audit output (especially annotation filenames, sample rows and ReadMe class mapping). We can then verify the data assumptions before you run conversion or training. Gemini can help explain/debug a cell, and I can review the code/output with you. Don't run Sections 5 onward until the class mapping and annotation schema are confirmed.